In [2]:
from google.colab import drive; drive.mount('/content/drive')
import sys, re, importlib, numpy as np, pandas as pd, torch
from scipy.stats import wilcoxon
from transformers import AutoTokenizer, AutoModelForSequenceClassification
CODE = "/content/drive/MyDrive/esci/code"; sys.path.insert(0, CODE)
import metrics, cross_encoder as C, train_cross_encoder as T
for m in (metrics, C, T): importlib.reload(m)
from metrics import evaluate
BASE = "/content/drive/MyDrive/esci"

df = pd.read_parquet(f"{BASE}/processed/rerank_pairs_v2.3.parquet")
qs = pd.read_parquet(f"{BASE}/dataset_v2.3/esci_electronics_v2.3_queries.parquet")[["query_id", "rule", "scope_group"]]
val = df[df.part == "val"].merge(qs, on="query_id", how="left").reset_index(drop=True); del df
CAT = {"mobile_computing": "Mobile & Computing", "components_accessories": "Components & Accessories",
       "home_electronics": "Home Electronics", "imaging": "Cameras & Imaging", "audio": "Audio",
       "computer_peripherals": "Computer Peripherals", "gaming": "Gaming", "smart_home": "Smart Home"}
val["category"] = np.where(val["rule"] == "A", val["scope_group"].map(CAT), None)
val["text_full"] = [C.product_to_text(C.esci_row_to_product(r), "full") for r in val.to_dict("records")]

# how each model's input is built (must match 08_grid Cell 1)
INPUT = {"raw_cat":     lambda q, c: C.query_to_text(q, category=c, use_rules=False),
         "focused_cat": lambda q, c: C.query_to_text(q, category=c)}
NOISE = {"clean": "", "price_number": " under 60000", "price_usd": " under $200", "price_rs": " below rs 15000",
         "cheap": " cheap", "platform": " on daraz", "delivery": " with free delivery"}
Q = val[["query_id", "query", "category"]].drop_duplicates("query_id")

# 1) rule coverage: did query_to_text remove the noise completely?
print("Rule coverage (focused text with noise == focused text without noise):")
clean_f = {r.query_id: INPUT["focused_cat"](r.query, r.category) for r in Q.itertuples()}
for n, s in NOISE.items():
    if n == "clean": continue
    same = np.mean([INPUT["focused_cat"](r.query + s, r.category) == clean_f[r.query_id] for r in Q.itertuples()])
    print(f"  {n:13s} {same:6.1%}")
ex = Q.iloc[0]
for n, s in NOISE.items():
    print(f"  e.g. {n:13s} raw_cat: {INPUT['raw_cat'](ex.query + s, ex.category)!r:60s} focused_cat: {INPUT['focused_cat'](ex.query + s, ex.category)!r}")

# 2) best epoch per run (normal formats only, same rule as before)
L = pd.read_csv(f"{BASE}/results/s7_grid_val_log.csv"); L = L[L["style"].isin(C.STYLES)]
def best(qcol):
    l = L[L.run.str.fullmatch(rf"minilm_l6__q_{qcol}(__s\d+)?")]
    ep = l.groupby(["run", "epoch"])["ndcg@10"].mean().reset_index()
    return ep.loc[ep.groupby("run")["ndcg@10"].idxmax()][["run", "epoch"]]
MODELS = {g: best(g) for g in INPUT}
for g, b in MODELS.items(): print(g, b.values.tolist())

# 3) score every model on every noise type
PQ, rows, texts = {}, [], val["text_full"].tolist()
for g, b in MODELS.items():
    for _, r in b.iterrows():
        path = f"{BASE}/models/{r.run}/epoch_{int(r.epoch)}"
        tok = AutoTokenizer.from_pretrained(path)
        m = AutoModelForSequenceClassification.from_pretrained(path).to("cuda").eval()
        for n, s in NOISE.items():
            qmap = {x.query_id: INPUT[g](x.query + s, x.category) for x in Q.itertuples()}
            v = val[["query_id", "gain"]].copy()
            v["score"] = T.score_pairs(m, tok, val["query_id"].map(qmap).tolist(), texts, 256, 128)
            mtr, pq = evaluate(v); rows.append({"model": g, "run": r.run, "noise": n, **mtr})
            PQ[(g, n)] = PQ.get((g, n), []) + [pq.set_index("query_id")["ndcg@10"]]
        del m; torch.cuda.empty_cache(); print("scored", r.run)

R = pd.DataFrame(rows)
tab = R.groupby(["noise", "model"])["ndcg@10"].mean().unstack("model").reindex(list(NOISE))
for g in INPUT: tab[f"drop_{g}"] = tab.loc["clean", g] - tab[g]
print("\n", tab.round(4).to_string())

# 4) paired test on the DROP (clean - noisy), per query, averaged over seeds
def per_q(g, n): return pd.concat(PQ[(g, n)], axis=1).mean(axis=1)
def drop(g, noises): return pd.concat([per_q(g, "clean") - per_q(g, n) for n in noises], axis=1).mean(axis=1)
def test(label, noises):
    d = drop("raw_cat", noises) - drop("focused_cat", noises); nz = d[d != 0]
    p = wilcoxon(nz).pvalue if len(nz) else float("nan")
    print(f"{label:13s}: raw_cat drop {drop('raw_cat', noises).mean():+.4f} | focused_cat drop {drop('focused_cat', noises).mean():+.4f} | p = {p:.4f}")
    return d.mean(), p
print()
noisy = [n for n in NOISE if n != "clean"]
for n in noisy: test(n, [n])
diff, p = test("ALL NOISE", noisy)
print("\nDECISION:", "query focusing HELPS ranking under noisy queries" if (diff > 0 and p < 0.05)
      else "no significant benefit from query focusing on this test")
R.to_csv(f"{BASE}/results/s9c_query_noise.csv", index=False)

Mounted at /content/drive
Rule coverage (focused text with noise == focused text without noise):
  price_number  100.0%
  price_usd     100.0%
  price_rs      100.0%
  cheap         100.0%
  platform      100.0%
  delivery      100.0%
  e.g. clean         raw_cat: '1 cordless telephone without answering capability'          focused_cat: '1 cordless telephone without answering capability'
  e.g. price_number  raw_cat: '1 cordless telephone without answering capability under 60000' focused_cat: '1 cordless telephone without answering capability'
  e.g. price_usd     raw_cat: '1 cordless telephone without answering capability under $200' focused_cat: '1 cordless telephone without answering capability'
  e.g. price_rs      raw_cat: '1 cordless telephone without answering capability below rs 15000' focused_cat: '1 cordless telephone without answering capability'
  e.g. cheap         raw_cat: '1 cordless telephone without answering capability cheap'    focused_cat: '1 cordless telephone with

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

scored minilm_l6__q_raw_cat


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

scored minilm_l6__q_raw_cat__s43


...

scored minilm_l6__q_focused_cat


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

scored minilm_l6__q_focused_cat__s43


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

scored minilm_l6__q_focused_cat__s44

 model         focused_cat  raw_cat  drop_raw_cat  drop_focused_cat
noise                                                             
clean              0.9278   0.9291        0.0000               0.0
price_number       0.9278   0.8912        0.0379               0.0
price_usd          0.9278   0.9241        0.0050               0.0
price_rs           0.9278   0.9034        0.0257               0.0
cheap              0.9278   0.9238        0.0053               0.0
platform           0.9278   0.8912        0.0379               0.0
delivery           0.9278   0.9251        0.0041               0.0

price_number : raw_cat drop +0.0379 | focused_cat drop +0.0000 | p = 0.0000
price_usd    : raw_cat drop +0.0050 | focused_cat drop +0.0000 | p = 0.0156
price_rs     : raw_cat drop +0.0257 | focused_cat drop +0.0000 | p = 0.0000
cheap        : raw_cat drop +0.0053 | focused_cat drop +0.0000 | p = 0.0005
platform     : raw_cat drop +0.0379 | focused_cat dro

In [3]:
a, b = per_q("raw_cat", "price_number"), per_q("raw_cat", "platform")
print(f"queries with identical NDCG: {(a.round(6) == b.round(6)).mean():.1%} | corr {a.corr(b):.4f}")
print(R[R.model == "raw_cat"].pivot(index="run", columns="noise", values="ndcg@10")[["price_number", "platform"]].round(5))
for r in Q.itertuples():
    for n, s in NOISE.items():
        if n != "clean" and INPUT["focused_cat"](r.query + s, r.category) != clean_f[r.query_id]:
            print(f"NOT CLEANED [{n}] {r.query!r} -> {INPUT['focused_cat'](r.query + s, r.category)!r}")

queries with identical NDCG: 44.4% | corr 0.8159
noise                      price_number  platform
run                                              
minilm_l6__q_raw_cat             0.8931    0.8943
minilm_l6__q_raw_cat__s43        0.8889    0.8865
minilm_l6__q_raw_cat__s44        0.8917    0.8929
